# CSE 654 — Bağlaç deneyi (sadece BERTurk, 4 seed)
1. **Bağlaç tuzak testi:** Test setindeki bağlaçlı zıt cümlelerden bağlaç silinir (target ve etiket aynı). Kontrol: bağlaç yerine anlamsız bir kelime silinir.
2. **Düzeltme:** Eğitimdeki bağlaçlı cümlelerin bağlaçsız kopyaları eğitime eklenir.
3. **Seed:** Hem temel hem düzeltilmiş BERTurk 4 seed ile (42, 43, 44, 45) eğitilir; sonuçlar ortalama ± standart sapma olarak verilir.

**Değişmeyenler:** veri, bölme (`split_ids.csv`), dev/test, eğitim ayarları (LR 2e-5, batch 32, 4 epoch, warmup %10, WD 0.01, fp16, en iyi epoch dev macro-F1 ile). Seed'ler arasında sadece eğitimdeki rastgelelik değişir.
Seed 42 modelleri daha önce aynı ayarlarla eğitildiği için tekrar eğitilmez, Drive'dan kullanılır.

**Sonuçlar:** `sonuclar/2_baglac_berturk/`. Eski bağlaç sonuç klasörleri (`2_baglac_tuzak_testi`, `3_duzeltme_baglac`) silinir. `1_zitlik_testi` ve modeller yerinde kalır.

Çalışma zamanı: **T4 GPU**. 6 eğitim var (~2,5–3,5 saat). Oturum kapanırsa baştan çalıştır; biten eğitimler atlanır, yarıda kalan checkpoint'ten devam eder.

## Drive ve klasörler

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
PROJE      = '/content/drive/MyDrive/cse654_absa'
VERI       = f'{PROJE}/absa_10k_final.csv'
SPLIT_FILE = f'{PROJE}/splits/split_ids.csv'
SONUC      = f'{PROJE}/sonuclar/2_baglac_berturk'
TAHMIN_DIR = f'{SONUC}/tahminler'
MODEL_ROOT = f'{PROJE}/berturk_baglac_seed'
ART_DIR    = f'{PROJE}/artirma/baglac'
for p in [VERI, SPLIT_FILE]:
    assert os.path.exists(p), f'Bulunamadı: {p}'

# Eski bağlaç sonuçlarını sil (bir kez)
for eski in ['2_baglac_tuzak_testi', '3_duzeltme_baglac']:
    yol = f'{PROJE}/sonuclar/{eski}'
    if os.path.isdir(yol):
        shutil.rmtree(yol); print('Silindi:', yol)
for p in [SONUC, TAHMIN_DIR, MODEL_ROOT, ART_DIR]:
    os.makedirs(p, exist_ok=True)
print('Hazır')

Mounted at /content/drive
Hazır


## Kütüphaneler

In [3]:
!pip -q install transformers accelerate datasets

## Ayarlar

In [4]:
import ast, re, math, inspect, gc, random
import numpy as np
import pandas as pd
import torch, transformers
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset
print('transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')

MODEL_ADI = 'dbmdz/bert-base-turkish-cased'
SEEDS = [42, 43, 44, 45]
LR, BATCH, EPOCHS, WARMUP, WD = 2e-5, 32, 4, 0.1, 0.01      # ilk eğitimle birebir aynı
VERI_SEED = 42                                              # veri/tuzak üretimi her seed'de aynı kalsın
# Seed 42 modelleri zaten var (aynı ayarlarla eğitildi)
HAZIR_42 = {'temel': f'{PROJE}/berturk_final', 'baglac': f'{PROJE}/duzeltme_baglac/berturk_final'}

LABELS = ['negative', 'neutral', 'positive']
label2id = {l: i for i, l in enumerate(LABELS)}; id2label = dict(enumerate(LABELS))
BAGLAC_RE = re.compile(r'\b(?:ne var ki|ama|fakat|ancak|lakin|oysa)\b', re.IGNORECASE)
ISLEV = {'bir', 've', 'bu', 'şu', 'için', 'gibi', 'ile', 'da', 'de', 'çok', 'en', 'daha', 'her', 'olarak', 'kadar'}
WC = '0-9A-Za-zÇĞİÖŞÜçğıöşüâîûÂÎÛș'
def birebir(text, t): return re.search(rf'(?<![{WC}]){re.escape(t)}(?![{WC}])', text) is not None
def baglac_sil(text): return re.sub(r'\s+', ' ', BAGLAC_RE.sub(' ', text)).strip()
def norm(s): return ' '.join(s.lower().split())

transformers 5.22.1 | GPU: Tesla T4


## Veri, tuzak cümleleri ve eğitim artırması

In [5]:
df = pd.read_csv(VERI, dtype=str, keep_default_na=False)
df['id'] = df['id'].astype(int)
df['targets'] = df['target'].map(ast.literal_eval); df['polarities'] = df['polarity'].map(ast.literal_eval)
def grup(l):
    if len(l) == 1: return 'tek_hedefli'
    if len(set(l)) == 1: return 'ayni_duygulu'
    if {'positive', 'negative'} <= set(l): return 'zit_duygulu'
    return 'notrle_karisik'
df['grup']   = df.polarities.map(grup)
df['baglac'] = df.text.map(lambda s: int(bool(BAGLAC_RE.search(s))))
df = df.merge(pd.read_csv(SPLIT_FILE), on='id'); assert df.split.notna().all()

def ciftler(d, col='text'):
    return pd.DataFrame([dict(id=r.id, sira=k, text=getattr(r, col), target=t, label=p, grup=r.grup, baglac=r.baglac)
                         for r in d.itertuples() for k, (t, p) in enumerate(zip(r.targets, r.polarities), start=1)])
train_df, dev_df, test_df = (ciftler(df[df.split == s]) for s in ['train', 'dev', 'test'])

# ---- 1) TUZAK CÜMLELERİ (sadece test) ----
rng = random.Random(VERI_SEED)
def islev_sil(text, targets):
    korunan = [(m.start(), m.end()) for t in targets for m in re.finditer(re.escape(t), text)]
    adaylar = [m for m in re.finditer(r'\S+', text)
               if m.group().lower() in ISLEV and not any(s < m.end() and m.start() < e for s, e in korunan)]
    if not adaylar: return None
    m = rng.choice(adaylar)
    return re.sub(r'\s+', ' ', text[:m.start()] + ' ' + text[m.end():]).strip()

ad = df[(df.split == 'test') & (df.grup == 'zit_duygulu') & (df.baglac == 1)].copy()
ad['text_baglacsiz'] = ad.text.map(baglac_sil)
ad['text_kontrol']   = [islev_sil(t, T) for t, T in zip(ad.text, ad.targets)]
ad = ad[[all(birebir(s, t) for t in T) for s, T in zip(ad.text_baglacsiz, ad.targets)]]
ad['text_kontrol'] = [s if isinstance(s, str) and all(birebir(s, t) for t in T) else None for s, T in zip(ad.text_kontrol, ad.targets)]
tuzak = pd.DataFrame([dict(id=r.id, sira=k, target=t, label=p, text_orijinal=r.text,
                           text_baglacsiz=r.text_baglacsiz, text_kontrol=r.text_kontrol)
                      for r in ad.itertuples() for k, (t, p) in enumerate(zip(r.targets, r.polarities), start=1)])
tuzak.to_csv(f'{SONUC}/tuzak_cumleleri.csv', index=False)
print(f'Tuzak: {ad.id.nunique()} cümle, {len(tuzak)} çift (kontrol versiyonu olan: {tuzak.text_kontrol.notna().sum()})')

# ---- 2) EĞİTİM ARTIRMASI (sadece train) ----
tr = df[(df.split == 'train') & (df.baglac == 1)].copy()
tr['text_yeni'] = tr.text.map(baglac_sil)
mevcut = set(df[df.split == 'train'].text.map(norm))
tr = tr[[all(birebir(s, t) for t in T) and norm(s) not in mevcut for s, T in zip(tr.text_yeni, tr.targets)]]
ek = ciftler(tr, 'text_yeni')
ek.to_csv(f'{ART_DIR}/train_baglacsiz_ekleri.csv', index=False)

# Sızıntı kontrolleri
assert not set(ek.id) & set(df[df.split != 'train'].id), 'Kopyalarda dev/test cümlesi var!'
assert not set(ek.text.map(norm)) & set(df[df.split != 'train'].text.map(norm)), 'Bir kopya dev/test cümlesiyle aynı!'
assert not set(tuzak.id) & set(df[df.split != 'test'].id), 'Tuzak cümlelerinde test dışı cümle var!'
print('Sızıntı kontrolleri geçti ✔')

EGITIM = {'temel':  train_df,
          'baglac': pd.concat([train_df, ek], ignore_index=True).sample(frac=1, random_state=VERI_SEED).reset_index(drop=True)}
print(f'Eğitim çiftleri — temel: {len(EGITIM["temel"]):,} | bağlaç: {len(EGITIM["baglac"]):,} (+{len(ek):,}) | dev {len(dev_df):,} | test {len(test_df):,}')

Tuzak: 265 cümle, 761 çift (kontrol versiyonu olan: 640)
Sızıntı kontrolleri geçti ✔
Eğitim çiftleri — temel: 19,715 | bağlaç: 24,632 (+4,917) | dev 2,168 | test 5,407


## Fonksiyonlar

In [6]:
tok = AutoTokenizer.from_pretrained(MODEL_ADI)
hepsi = pd.concat([train_df, dev_df, test_df])
lens = np.array([len(x) for x in tok(hepsi.text.tolist(), hepsi.target.tolist())['input_ids']])
MAX_LEN = int(min(512, math.ceil(np.percentile(lens, 99.9) / 32) * 32))      # ilk eğitimle aynı kural
print('MAX_LEN =', MAX_LEN)

def hazirla(dfp):
    ds = Dataset.from_pandas(dfp[['text', 'target', 'label']].reset_index(drop=True))
    def enc(b):
        out = tok(b['text'], b['target'], truncation='only_first', max_length=MAX_LEN)
        out['labels'] = [label2id[x] for x in b['label']]
        return out
    return ds.map(enc, batched=True, remove_columns=['text', 'target', 'label'])

def compute_metrics(p):
    y_hat, y = p.predictions.argmax(-1), p.label_ids
    return {'accuracy': accuracy_score(y, y_hat), 'macro_f1': f1_score(y, y_hat, average='macro')}

def egit(train_ds, dev_ds, seed, out_dir):
    transformers.set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ADI, num_labels=3, id2label=id2label, label2id=label2id)
    kw = dict(output_dir=f'{out_dir}/checkpoints', save_strategy='epoch', save_total_limit=2,
              load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
              learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
              num_train_epochs=EPOCHS, warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * EPOCHS),
              weight_decay=WD, fp16=torch.cuda.is_available(), logging_steps=50, report_to='none', seed=seed)
    kw['eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters else 'evaluation_strategy'] = 'epoch'
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    last = get_last_checkpoint(kw['output_dir']) if os.path.isdir(kw['output_dir']) else None
    print('Checkpoint bulundu, devam:' if last else 'Sıfırdan eğitim.', last or '')
    trainer.train(resume_from_checkpoint=last)
    trainer.save_model(f'{out_dir}/final'); tok.save_pretrained(f'{out_dir}/final')
    shutil.rmtree(kw['output_dir'], ignore_errors=True)        # yer kaplamasın; final model kaldı
    return trainer.model

@torch.no_grad()
def tahmin(model, metinler, hedefler, bs=64):
    model.eval(); out = []
    for i in range(0, len(metinler), bs):
        enc = tok(list(metinler[i:i+bs]), list(hedefler[i:i+bs]), truncation='only_first', max_length=512,
                  padding=True, return_tensors='pt').to(model.device)
        with torch.autocast('cuda', enabled=torch.cuda.is_available()):
            out += model(**enc).logits.argmax(-1).tolist()
    return [id2label[i] for i in out]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

MAX_LEN = 96


## Eğitimler ve tahminler (temel + bağlaç, 4 seed)

In [7]:
dev_ds = hazirla(dev_df)
for varyant in ['temel', 'baglac']:
    train_ds = None
    for seed in SEEDS:
        ad_ = f'{varyant}_s{seed}'
        f_test, f_tuzak = f'{TAHMIN_DIR}/{ad_}_test.csv', f'{TAHMIN_DIR}/{ad_}_tuzak.csv'
        if os.path.exists(f_test) and os.path.exists(f_tuzak):
            print(f'[{ad_}] bitti, atlanıyor'); continue
        print(f'\n===== {ad_} =====')
        if seed == 42 and os.path.isdir(HAZIR_42[varyant]):
            print('Seed 42 modeli Drive\'dan kullanılıyor:', HAZIR_42[varyant])
            model = AutoModelForSequenceClassification.from_pretrained(HAZIR_42[varyant]).to('cuda' if torch.cuda.is_available() else 'cpu')
        elif os.path.isdir(f'{MODEL_ROOT}/{ad_}/final'):
            model = AutoModelForSequenceClassification.from_pretrained(f'{MODEL_ROOT}/{ad_}/final').to('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            train_ds = train_ds or hazirla(EGITIM[varyant])
            model = egit(train_ds, dev_ds, seed, f'{MODEL_ROOT}/{ad_}')

        t = test_df.copy(); t['pred'] = tahmin(model, t.text.tolist(), t.target.tolist())
        z = tuzak.copy()
        z['orijinal']  = tahmin(model, z.text_orijinal.tolist(),  z.target.tolist())
        z['baglacsiz'] = tahmin(model, z.text_baglacsiz.tolist(), z.target.tolist())
        k = z.text_kontrol.notna(); z['kontrol'] = pd.Series([None] * len(z), dtype=object)
        z.loc[k, 'kontrol'] = tahmin(model, z.loc[k, 'text_kontrol'].tolist(), z.loc[k, 'target'].tolist())
        t.to_csv(f_test, index=False); z.to_csv(f_tuzak, index=False)
        print(f'test acc {(t.pred == t.label).mean():.4f} | tuzak değişen %{100 * (z.orijinal != z.baglacsiz).mean():.2f}')
        del model; gc.collect(); torch.cuda.empty_cache()

Map:   0%|          | 0/2168 [00:00<?, ? examples/s]

[temel_s42] bitti, atlanıyor

===== temel_s43 =====


Map:   0%|          | 0/19715 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (58 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

test acc 0.8648 | tuzak değişen %4.20


## Sonuçlar: ortalama ± standart sapma

In [8]:
def olc(t, z):
    t = t.assign(dogru=(t.pred == t.label).astype(int)); zit = t[t.grup == 'zit_duygulu']; k = z.kontrol.notna()
    return pd.Series({
        'test: accuracy (tümü)':          t.dogru.mean(),
        'test: macro-F1 (tümü)':          f1_score(t.label, t.pred, labels=LABELS, average='macro'),
        'test: aynı duygulu accuracy':    t[t.grup == 'ayni_duygulu'].dogru.mean(),
        'test: zıt duygulu accuracy':     zit.dogru.mean(),
        'test: zıt cümle tam doğru':      zit.groupby('id').dogru.min().mean(),
        'test: doğal bağlaçsız zıt acc':  zit[zit.baglac == 0].dogru.mean(),
        'test: bağlaçlı zıt acc':         zit[zit.baglac == 1].dogru.mean(),
        'tuzak: acc orijinal':            (z.orijinal == z.label).mean(),
        'tuzak: acc bağlaçsız':           (z.baglacsiz == z.label).mean(),
        'tuzak: değişen (%)':             100 * (z.orijinal != z.baglacsiz).mean(),
        'tuzak: doğrudan yanlışa (%)':    100 * ((z.orijinal == z.label) & (z.baglacsiz != z.label)).mean(),
        'tuzak KONTROL: değişen (%)':     100 * (z.loc[k, 'orijinal'] != z.loc[k, 'kontrol']).mean(),
    })

sonuc = {}
for v in ['temel', 'baglac']:
    for s in SEEDS:
        ft, fz = f'{TAHMIN_DIR}/{v}_s{s}_test.csv', f'{TAHMIN_DIR}/{v}_s{s}_tuzak.csv'
        if os.path.exists(ft) and os.path.exists(fz):
            sonuc[(v, s)] = olc(pd.read_csv(ft), pd.read_csv(fz))
seed_tablo = pd.DataFrame(sonuc).round(4)
seed_tablo.to_csv(f'{SONUC}/sonuc_seed_bazinda.csv')

ortak = sorted({s for v, s in sonuc if v == 'temel'} & {s for v, s in sonuc if v == 'baglac'})
T = pd.DataFrame({s: sonuc[('temel', s)] for s in ortak}); B = pd.DataFrame({s: sonuc[('baglac', s)] for s in ortak})
fmt = lambda d: d.mean(axis=1).round(4).astype(str) + ' ± ' + d.std(axis=1, ddof=1).round(4).astype(str)
ozet = pd.DataFrame({'temel (ort ± std)': fmt(T), 'bağlaç düzeltmeli (ort ± std)': fmt(B),
                     'fark (ort ± std)': fmt(B - T),
                     'kaç seed\'de iyileşti': [f'{int(((B - T).loc[m] * (-1 if "değişen" in m or "yanlışa" in m else 1) > 0).sum())}/{len(ortak)}' for m in T.index]})
print(f'Seed sayısı: {len(ortak)} {ortak}')
display(ozet)
ozet.to_csv(f'{SONUC}/sonuc_ozet.csv')
print('Kaydedildi:', SONUC)

Seed sayısı: 4 [42, 43, 44, 45]


,temel (ort ± std),bağlaç düzeltmeli (ort ± std),fark (ort ± std),kaç seed'de iyileşti
test: accuracy (tümü),0.8775 ± 0.0039,0.8737 ± 0.0063,-0.0038 ± 0.0102,2/4
test: macro-F1 (tümü),0.8608 ± 0.0047,0.8589 ± 0.0058,-0.0019 ± 0.0103,2/4
test: aynı duygulu accuracy,0.9326 ± 0.0037,0.9258 ± 0.0029,-0.0067 ± 0.0064,0/4
test: zıt duygulu accuracy,0.8672 ± 0.0067,0.8553 ± 0.015,-0.0119 ± 0.0188,1/4
test: zıt cümle tam doğru,0.7092 ± 0.011,0.6869 ± 0.018,-0.0223 ± 0.0198,0/4
test: doğal bağlaçsız zıt acc,0.8125 ± 0.0094,0.8052 ± 0.0242,-0.0073 ± 0.0324,3/4
test: bağlaçlı zıt acc,0.8968 ± 0.0068,0.8824 ± 0.0103,-0.0145 ± 0.0124,0/4
tuzak: acc orijinal,0.8968 ± 0.0068,0.8824 ± 0.0103,-0.0145 ± 0.0124,0/4
tuzak: acc bağlaçsız,0.8624 ± 0.0051,0.8683 ± 0.0156,0.0059 ± 0.0153,3/4
tuzak: değişen (%),5.8804 ± 0.2709,4.1064 ± 0.6733,-1.774 ± 0.8345,4/4


Kaydedildi: /content/drive/MyDrive/cse654_absa/sonuclar/2_baglac_berturk
